# Capstone — Refresh / Content Opportunity Scoring

**Research question:** Can pre-decision Google Search Console signals identify pages that should be reviewed first for possible search-visibility decline, while remaining useful on clients that were not seen during training?

This capstone comes directly from the **FlyRank Refresh / Content Opportunity Scoring** lane: when a content team has far more pages than it can manually review, the practical problem is deciding **which pages deserve attention first** without turning noisy search signals into overconfident recommendations. I use the FlyRank internship warehouse to test whether a small, leakage-safe set of pre-decision signals can support that triage. Every score is treated as **directional decision-support**, never as proof of Google's ranking algorithm or proof that refreshing content causes recovery.


## 1. Question

The FlyRank content problem behind this lane is operational triage: a content team cannot inspect every page after every movement in search visibility, so it needs a defensible way to decide which pages should be reviewed first.

The target is a simple forward proxy: `decline_proxy = 1` when a page's March 16–31 impressions are below 80% of its March 1–15 impressions. The model must only use information available by March 15. The output is therefore a **review-priority signal**, not an automatic instruction to rewrite, refresh, redirect, or publish.


## 2. Data

- **Release:** FlyRank internship warehouse hosted on Hugging Face.
- **Primary table:** `fact_content_daily_performance`.
- **Development window:** March 2026.
- **Feature window:** March 1–15, 2026.
- **Outcome window:** March 16–31, 2026.
- **Modeling grain:** one row per `client_hash_id × content_hash_id`.
- **Observed modeling frame in the executed run:** 151,981 content rows across 44 clients.
- **Observed decline-proxy rate:** 0.327.

Only rows with `gsc_data_available IS TRUE` are used. Client/content IDs are grouping/context fields, not predictors. Second-half metrics and future-derived fields are excluded from features.


In [1]:
# Rebuild the data from the gated warehouse when reproducing in Colab.
# Add HF_TOKEN as a Colab Secret or environment variable. Never commit the token.
%pip -q install duckdb huggingface_hub scikit-learn pandas numpy matplotlib

import os, getpass, duckdb, pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, balanced_accuracy_score, precision_score, recall_score

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Enter Hugging Face READ token (hidden): ")

con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [HF_TOKEN])
REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

FEATURE_COLS = [
    "first_half_impressions","first_half_clicks","first_half_avg_position",
    "first_half_active_days","first_half_ctr_pct"
]
TARGET = "decline_proxy"
GROUP = "client_hash_id"

feature_sql = f'''
WITH daily AS (
  SELECT report_date, client_hash_id, content_hash_id,
         COALESCE(gsc_impressions,0) AS gsc_impressions,
         COALESCE(gsc_clicks,0) AS gsc_clicks,
         NULLIF(gsc_avg_position,0) AS gsc_avg_position
  FROM {DAILY}
  WHERE gsc_data_available IS TRUE
),
agg AS (
  SELECT client_hash_id, content_hash_id,
         SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                  THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
         SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                  THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
         AVG(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                  THEN gsc_avg_position END) AS first_half_avg_position,
         COUNT(DISTINCT CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                  AND gsc_impressions > 0 THEN report_date END) AS first_half_active_days,
         SUM(CASE WHEN report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
                  THEN gsc_impressions ELSE 0 END) AS second_half_impressions
  FROM daily GROUP BY 1,2
)
SELECT client_hash_id, content_hash_id,
       first_half_impressions, first_half_clicks, first_half_avg_position,
       first_half_active_days,
       100.0*first_half_clicks/NULLIF(first_half_impressions,0) AS first_half_ctr_pct,
       CASE WHEN second_half_impressions < 0.80*first_half_impressions THEN 1 ELSE 0 END AS decline_proxy
FROM agg
WHERE first_half_impressions > 0
'''
model_df = con.sql(feature_sql).df()
print(model_df.shape, model_df[GROUP].nunique(), model_df[TARGET].mean())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(151981, 8) 44 0.3268369072449846


## 3. Methodology

Five pre-decision features are used: impressions, clicks, average position, active days, and CTR from March 1–15.

The learned model is regularized Logistic Regression with median imputation, standardization, and class balancing. The transparent baseline awards review-priority points for training-distribution quartile conditions: low CTR, weak average position, few active days, few clicks, and low impressions.

The decisive validation is a **client-grouped 80/20 holdout** using `GroupShuffleSplit`, so no client appears in both training and test. Precision@50 is the primary metric because the product output is a ranked queue. PR-AUC, ROC-AUC, balanced accuracy, precision, recall, and the test base rate are reported as supporting metrics.

Leakage controls:
1. Features end on March 15; the label starts on March 16.
2. Client/content IDs are never predictors.
3. Future-window metrics are excluded.
4. A deliberately leaked future trend variable was used only as a sanity test in earlier work and removed.


In [2]:
# Client-grouped split + baseline + model.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(model_df[FEATURE_COLS], model_df[TARGET], groups=model_df[GROUP]))
train_df, test_df = model_df.iloc[train_idx].copy(), model_df.iloc[test_idx].copy()
assert set(train_df[GROUP]).isdisjoint(set(test_df[GROUP]))

cut = {
    "ctr_q25": train_df["first_half_ctr_pct"].quantile(.25),
    "position_q75": train_df["first_half_avg_position"].quantile(.75),
    "active_q25": train_df["first_half_active_days"].quantile(.25),
    "clicks_q25": train_df["first_half_clicks"].quantile(.25),
    "impressions_q25": train_df["first_half_impressions"].quantile(.25),
}
def baseline_score(frame):
    s = pd.Series(0.0,index=frame.index)
    s += (frame["first_half_ctr_pct"] <= cut["ctr_q25"]).astype(float)
    s += (frame["first_half_avg_position"] >= cut["position_q75"]).astype(float)
    s += (frame["first_half_active_days"] <= cut["active_q25"]).astype(float)
    s += (frame["first_half_clicks"] <= cut["clicks_q25"]).astype(float)
    s += (frame["first_half_impressions"] <= cut["impressions_q25"]).astype(float)
    return s

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
model.fit(train_df[FEATURE_COLS], train_df[TARGET])
model_prob = model.predict_proba(test_df[FEATURE_COLS])[:,1]
baseline_prob = baseline_score(test_df)

def precision_at_k(y, score, k=50):
    y, score = np.asarray(y), np.asarray(score)
    idx = np.argsort(-score, kind="mergesort")[:min(k,len(y))]
    return float(y[idx].mean())

print("grouped test rows:", len(test_df))
print("grouped test clients:", test_df[GROUP].nunique())
print("grouped base rate:", test_df[TARGET].mean())
print("model Precision@50:", precision_at_k(test_df[TARGET], model_prob))
print("baseline Precision@50:", precision_at_k(test_df[TARGET], baseline_prob))


grouped test rows: 13210
grouped test clients: 9
grouped base rate: 0.37335352006056016
model Precision@50: 0.4
baseline Precision@50: 0.52


## 4. Results — model vs baseline

On the Week-5 client-held-out split, the **transparent rule baseline beat Logistic Regression on the primary ranking metric**:

| Method | Precision@50 | PR-AUC | ROC-AUC | Balanced accuracy | Precision | Recall |
|---|---:|---:|---:|---:|---:|---:|
| Rule baseline | **0.540** | 0.395 | 0.543 | 0.523 | 0.399 | 0.464 |
| Logistic Regression | 0.380 | 0.395 | 0.549 | 0.528 | 0.397 | 0.575 |

The learned model therefore did **not** earn a claim of improvement over the simple baseline.

A second validation audit showed an even more important result: the same Logistic Regression scored **Precision@50 = 0.620** under a naive random-row split but only **0.340** under a client-grouped split. The grouped test base rate was **0.373**, so the model's top-50 ranking was below the held-out base rate in that audit.

This is the capstone's central finding: a flattering random split can materially overstate the usefulness of a search-opportunity model when client-specific patterns leak across train/test boundaries.


In [3]:
# Frozen public-safe receipt from the executed March 2026 notebooks.
results = pd.DataFrame([
    ["Rule baseline",0.540,0.395,0.543,0.523,0.399,0.464],
    ["Logistic Regression",0.380,0.395,0.549,0.528,0.397,0.575],
], columns=["method","precision_at_50","pr_auc","roc_auc","balanced_accuracy","precision","recall"])
display(results)

validation = pd.DataFrame([
    ["Naive random row split",0.327,0.620,0.413,0.597],
    ["Client-grouped split",0.373,0.340,0.395,0.549],
], columns=["split","base_rate","precision_at_50","pr_auc","roc_auc"])
display(validation)


,method,precision_at_50,pr_auc,roc_auc,balanced_accuracy,precision,recall
0,Rule baseline,0.54,0.395,0.543,0.523,0.399,0.464
1,Logistic Regression,0.38,0.395,0.549,0.528,0.397,0.575


,split,base_rate,precision_at_50,pr_auc,roc_auc
0,Naive random row split,0.327,0.62,0.413,0.597
1,Client-grouped split,0.373,0.34,0.395,0.549


## 5. Limitations & honest framing

- The study uses one March 2026 decision window; it does not establish stability across future months.
- The warehouse is an unbalanced panel and clients have different tracking histories.
- The label is a simple visibility-decline proxy, not a causal diagnosis.
- Search movement can reflect seasonality, SERP changes, site changes, demand shifts, technical issues, or other unobserved factors.
- The learned model did not show useful grouped Precision@50 lift in the validation audit.
- The recommendation layer is therefore a **structured triage playbook**, not autonomous optimization.

Appropriate wording: **observed**, **measured**, **directional**, **decision-support**, **review candidate**.

Inappropriate wording: “predicts Google,” “proves ranking factors,” “refreshing causes recovery,” or “the model knows which pages Google will demote.”


## 6. Ranked recommendations

The action layer combines model risk with page value and transparent reason codes. In the held-out queue of **13,210** rows:

- **12,755 — Monitor / low-evidence**
- **285 — CTR opportunity**
- **170 — Ranking-depth opportunity**

The top-ranked public-safe action sequence is:

1. **Ranking-depth review** — prioritize high-impression pages with weak average position; inspect competing results, topical gaps, and internal linking before changing content.
2. **CTR review** — prioritize visible pages with very weak CTR; inspect query/intent fit, title/snippet alignment, and SERP presentation.
3. **Refresh review only when evidence supports it** — check factual staleness, cannibalization, usefulness, and content age/context; do not treat “refresh” as a causal fix.
4. **Monitor low-evidence pages** — defer edits when opportunity is small or signals are mixed.

The playbook deliberately defaults most pages to monitoring rather than forcing an action.


## 7. Artifacts the paper embeds

The deployed paper uses three aggregate, public-safe charts:

- `work/figures/model_vs_baseline.png`
- `work/figures/validation_sensitivity.png`
- `work/figures/action_mix.png`

No client names, domains, URLs, private queries, or row-level identifiers are published.


## 8. Reproducibility

Repository: https://github.com/kbhutto256/flyrank-ml-internship

Key notebooks:
- `work/notebooks/w03_data_contract.ipynb`
- `work/notebooks/w05_model.ipynb`
- `work/notebooks/w06_validation_audit.ipynb`
- `work/notebooks/w07_action_playbook.ipynb`
- `work/notebooks/capstone.ipynb`

Re-running the warehouse cells requires authorized read access to the gated FlyRank Hugging Face dataset and a private `HF_TOKEN`. Credentials must never be committed.


## 9. Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset**. Data source and internship program: https://flyrank.ai

The analysis uses anonymized/hash-based identifiers only and follows the public-release rule: no client names, domains, URLs, private queries, credentials, or raw exports are published.


## ML-12 closeout

### 5-minute demo outline

**0:00–0:45 — Question / FlyRank content problem**  
Open with the real lane problem: FlyRank's content workflow can surface many pages with changing search performance, but a content team has limited review capacity. My question was: **can pre-decision GSC signals help rank which pages deserve human review first, including for clients not seen during training?** Make clear that this is prioritization, not a claim about Google's algorithm or a causal promise that a refresh will recover traffic.

**0:45–1:45 — Method**  
Show the March timeline: features from **March 1–15** and the decline proxy from **March 16–31**. Explain the five features—impressions, clicks, average position, active days, and CTR—and the comparison between a transparent rule baseline and Logistic Regression. Emphasize the client-grouped holdout so no client's rows appear in both train and test, plus the leakage checks that remove IDs and future-window fields from predictors.

**1:45–2:45 — One chart**  
Show **`validation_sensitivity.png`**. Point to the contrast between a naive random-row split and client-grouped validation: random Precision@50 was **0.620**, while grouped Precision@50 fell to **0.340**. Explain that the grouped held-out base rate was **0.373**, so the flattering random result did not survive the stricter design.

**2:45–3:45 — One honest result**  
State the result plainly: the learned model did **not** earn a model-win claim. On the Week-5 grouped split, the transparent rule baseline achieved **Precision@50 = 0.540** versus **0.380** for Logistic Regression. The most useful discovery was therefore about **validation design**, not model complexity: client overlap can make a search-opportunity model look materially stronger than it is on unseen clients.

**3:45–4:45 — One recommendation**  
Recommend a conservative human-review playbook. Prioritize ranking-depth review for high-impression pages with weak position, CTR review for visible pages with weak CTR, and consider refresh only when separate evidence supports staleness or usefulness concerns. Keep mixed/low-evidence pages in monitoring instead of forcing an edit.

**4:45–5:00 — Close**  
Close with one sentence: **the capstone turned a ranking model experiment into a safer content-triage workflow because stricter validation changed what I was willing to claim and automate.**

### Social-post cut
I finished my FlyRank Search Intelligence capstone on content-opportunity scoring. The most useful result was not a flashy model win: a naive random split produced **Precision@50 = 0.620**, but client-grouped validation fell to **0.340**, showing how easily search models can look stronger when client patterns leak across train and test. I turned that finding into a conservative review playbook for CTR, ranking-depth, refresh, and monitoring decisions—framed as human-reviewed decision-support, not a claim about Google's algorithm.

### Employer-facing summary
I built a search-content prioritization workflow on the **FlyRank internship warehouse**, using DuckDB and scikit-learn with time-separated features and labels, leakage checks, a transparent baseline, Logistic Regression, and client-grouped validation. The analysis showed that a naive random split materially overstated ranking quality, and on the key grouped split the simple rule baseline outperformed Logistic Regression on Precision@50. I converted that result into an explainable human-review playbook for ranking-depth, CTR, refresh, and monitoring decisions instead of overstating model performance.
